# Create NEB Path Materials

Build initial → (optional intermediates) → final images for a Nudged Elastic Band path, and write them to a subfolder under `uploads/` in path order for [`create_materials_set.ipynb`](create_materials_set.ipynb).

Order is preserved by **numbering material names** (`00_...`, `01_...`, …): `create_materials_set.ipynb` (and `load_materials_from_folder`) sort by filename, and filenames come from material names.

## Usage

1. Set material and subfolder name in cell 1.2, transform params in 1.3.
1. Run all cells to build and write the path materials.
1. Open [`create_materials_set.ipynb`](create_materials_set.ipynb), set the same `SUBFOLDER_NAME` and `IS_ORDERED = True`, and run it to save the materials and create the platform set.
1. Use the printed set name as `MATERIAL_SET` in [`neb.ipynb`](workflows/neb.ipynb).

## Summary

1. Install packages and set parameters.
1. Load the starting material.
1. Clone as the initial image; transform a copy into the final image (default: translate one atom).
1. Name members in path order and write them to `uploads/<SUBFOLDER_NAME>/`.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")


### 1.2. Set parameters


In [ ]:
# Starting material (uploads folder or Standata name match)
FOLDER = "uploads"
MATERIAL_NAME = "Silicon"

# Subfolder under uploads/ to write path materials into — use the same value as
# SUBFOLDER_NAME in create_materials_set.ipynb.
SUBFOLDER_NAME = "neb_silicon"


### 1.3. Set example transformation parameters
Default example: translate one atom in crystal coordinates. Replace this cell and 2.3 below for other paths.


In [ ]:
ATOM_INDEX = 0
TRANSLATION = [0.0, 0.0, 0.1]


## 2. Build path materials
### 2.1. Load starting material


In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

source_material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME)
)
visualize(source_material)


### 2.2. Clone as the initial image


In [ ]:
initial_material = source_material.clone()
visualize(initial_material)


### 2.3. Transform into the final image (example)

Default: translate the atom at `ATOM_INDEX` by `TRANSLATION`. Replace with any other transformation (defects, swaps, `create_perturbation`, custom coordinates, …) — only the resulting list of materials in 2.4 matters.


In [ ]:
final_material = initial_material.clone()
coordinates = [list(coordinate) for coordinate in final_material.coordinates_array]
for axis_index, delta in enumerate(TRANSLATION):
    coordinates[ATOM_INDEX][axis_index] += delta
final_material.set_coordinates(coordinates)

# Optional: localized ∆z via Made create_perturbation (scalar f(x,y,z) → ∆z).
# import sympy as sp
# from mat3ra.made.tools.helpers import create_perturbation
#
# RADIUS_SIGMA = 0.05
# x, y, z = sp.symbols("x y z")
# center_x, center_y, center_z = initial_material.coordinates_array[ATOM_INDEX]
# weight = sp.exp(
#     -((x - center_x) ** 2 + (y - center_y) ** 2 + (z - center_z) ** 2) / (2 * RADIUS_SIGMA**2)
# )
# final_material = create_perturbation(
#     initial_material,
#     TRANSLATION[2] * weight,
#     use_cartesian_coordinates=False,
# )

print(
    f"Atom {ATOM_INDEX}: {initial_material.coordinates_array[ATOM_INDEX]} → "
    f"{final_material.coordinates_array[ATOM_INDEX]}"
)
visualize([initial_material, final_material])


### 2.4. Name members in path order and write to the subfolder

Numeric prefixes control load order in `create_materials_set.ipynb` (filenames are sorted; filenames come from material names).


In [ ]:
from mat3ra.notebooks_utils.material import set_materials
from mat3ra.notebooks_utils.settings import UPLOADS_FOLDER

path_materials = [initial_material, final_material]
for index, material in enumerate(path_materials):
    material.name = f"{index:02d}_{source_material.name}"

subfolder_path = f"{UPLOADS_FOLDER}/{SUBFOLDER_NAME}"
set_materials(path_materials, folder_path=subfolder_path)
print(f"Wrote {len(path_materials)} material(s) to {subfolder_path}/")
